# 🔢 03: Embeddings와 Vector Search

---

| 항목 | 내용 |
|------|------|
| **목표** | LLM이 내 문서를 모른다는 문제를 해결하기 위해 임베딩과 벡터 검색을 직관적으로 이해한다 |
| **예상 실행 시간** | ⏱️ 빠른 시연 5분 / 전체 15분 (모델 다운로드 포함) |
| **API 키** | ❌ 불필요 (sentence-transformers로 로컬 실행 가능) |
| **이전 노트북과의 연결** | LLM이 인터페이스가 됐지만, 내 회사 문서를 모른다. 이 문제를 해결하기 위해 검색 계층이 필요하다. |

---

## 🎯 핵심 메시지

> **LLM은 내 문서를 모릅니다.**  
> **Vector DB는 신비한 것이 아닙니다. 의미 기반 검색 인프라입니다.**

```
문제: "API 키 유효기간이 몇 일이에요?" → LLM은 모름
해결: 1) 문서를 벡터로 변환 (인덱싱)
     2) 질문도 벡터로 변환
     3) 가장 가까운 문서를 찾음 (검색)
     4) 찾은 문서 + 질문을 LLM에 넘김 (RAG)
```

In [ ]:
!pip install -q sentence-transformers rank-bm25 openai
print("✅ 설치 완료")

## 1️⃣ 환경 설정

In [ ]:
import os
import numpy as np
import pandas as pd
from IPython.display import display, HTML

# .env 에서 API 키 로드 (python-dotenv 필요, Colab 에서는 직접 입력 가능)
try:
    from dotenv import load_dotenv
    from pathlib import Path
    for _p in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
        if (_p / ".env").exists():
            load_dotenv(_p / ".env"); break
except ImportError:
    pass

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")

def setup(api_key=""):
    key = api_key or os.environ.get("OPENAI_API_KEY", "")
    if key and key not in ("", "sk-..."):
        try:
            from openai import OpenAI
            c = OpenAI(api_key=key)
            print("✅ API 모드 (text-embedding-3-small)")
            return "api", c
        except:
            pass
    print("✅ 로컬 모드 (all-MiniLM-L6-v2)")
    print("   💡 이 노트북은 API 없이도 완전히 실행됩니다!")
    return "local", None

MODE, client = setup(OPENAI_API_KEY)

# ──────────────────────────────────────────────
# 임베딩 헬퍼
# ──────────────────────────────────────────────
_st_model = None

def embed_texts(texts, mode=None, client=None):
    """텍스트 리스트를 numpy 벡터 배열로 임베딩합니다."""
    global _st_model
    m = mode or MODE
    
    if m == "api" and client:
        try:
            vecs = []
            for i in range(0, len(texts), 50):
                resp = client.embeddings.create(
                    input=texts[i:i+50],
                    model="text-embedding-3-small"
                )
                vecs.extend([r.embedding for r in resp.data])
            return np.array(vecs, dtype=np.float32)
        except Exception as e:
            print(f"⚠️ API 실패 → 로컬: {e}")
    
    # 로컬 임베딩
    if _st_model is None:
        print("📥 임베딩 모델 로딩 (all-MiniLM-L6-v2, 약 90MB)...")
        from sentence_transformers import SentenceTransformer
        _st_model = SentenceTransformer("all-MiniLM-L6-v2")
        print("✅ 완료")
    return _st_model.encode(texts, convert_to_numpy=True,
                            show_progress_bar=False).astype(np.float32)

def cosine_sim(query_vec, doc_vecs):
    """코사인 유사도 계산."""
    q = np.array(query_vec, dtype=np.float32).flatten()
    D = np.array(doc_vecs, dtype=np.float32)
    q_norm = np.linalg.norm(q)
    if q_norm < 1e-9:
        return np.zeros(len(D))
    d_norms = np.linalg.norm(D, axis=1)
    d_norms = np.where(d_norms < 1e-9, 1e-9, d_norms)
    return (D @ q) / (d_norms * q_norm)

print(f"\n준비 완료! 모드: {MODE}")

## 2️⃣ 문서 데이터 준비

In [ ]:
import sys
from pathlib import Path


def _ensure_project_root_on_path() -> None:
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    try:
        candidates.extend(path for path in cwd.iterdir() if path.is_dir())
    except OSError:
        pass

    for candidate in candidates:
        if (candidate / "helpers" / "sample_data.py").exists():
            candidate_str = str(candidate)
            if candidate_str not in sys.path:
                sys.path.insert(0, candidate_str)
            return

    raise ModuleNotFoundError(
        "프로젝트 루트를 찾지 못했습니다. `ai_special_course` 저장소를 clone 한 뒤 "
        "repo root 또는 notebooks 디렉터리에서 노트북을 실행하세요."
    )


_ensure_project_root_on_path()

# 테크코어 내부 문서 (00번 노트북과 동일한 데이터)
from helpers.sample_data import SAMPLE_DOCS_MINI_03 as SAMPLE_DOCS
# Colab 사용 시: !git clone <repo> 후 sys.path 에 추가하거나, sample_data.py 를 /content 에 업로드하세요.

print(f"✅ 문서 {len(SAMPLE_DOCS)}개 준비")
df_docs = pd.DataFrame(SAMPLE_DOCS)
display(df_docs[["doc_id", "category"]].assign(
    내용길이=df_docs["content"].str.len()
))

## 3️⃣ 임베딩이란 무엇인가?

**임베딩 = 텍스트를 숫자 벡터로 변환하는 것**

```
"API 키 보안"     → [0.23, -0.45, 0.87, ... (384차원)]
"시크릿 관리 방법" → [0.21, -0.43, 0.89, ... (의미가 비슷하면 가까운 벡터)]
"점심 메뉴 추천"   → [-0.51, 0.32, -0.14, ... (의미가 다르면 먼 벡터)]
```

핵심: **의미가 비슷한 텍스트 = 비슷한 방향의 벡터**

In [ ]:
import sys
from pathlib import Path


def _ensure_project_root_on_path() -> None:
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    try:
        candidates.extend(path for path in cwd.iterdir() if path.is_dir())
    except OSError:
        pass

    for candidate in candidates:
        if (candidate / "helpers" / "sample_data.py").exists():
            candidate_str = str(candidate)
            if candidate_str not in sys.path:
                sys.path.insert(0, candidate_str)
            return

    raise ModuleNotFoundError(
        "프로젝트 루트를 찾지 못했습니다. `ai_special_course` 저장소를 clone 한 뒤 "
        "repo root 또는 notebooks 디렉터리에서 노트북을 실행하세요."
    )


_ensure_project_root_on_path()

# 임베딩 직접 확인해보기
from helpers.sample_data import TEST_TEXTS_03 as test_texts
# Colab 사용 시: !git clone <repo> 후 sys.path 에 추가하거나, sample_data.py 를 /content 에 업로드하세요.

print("📐 임베딩 생성 중...")
test_vecs = embed_texts(test_texts)
print(f"벡터 형태: {test_vecs.shape} (문서 수 × 차원)")
print(f"각 벡터의 첫 5개 숫자 (예시):")
print(f"  '{test_texts[0][:20]}...': {test_vecs[0][:5].tolist()}")
print()

# 유사도 행렬 계산
n = len(test_texts)
sim_matrix = np.zeros((n, n))
for i in range(n):
    scores = cosine_sim(test_vecs[i], test_vecs)
    sim_matrix[i] = scores

# 유사도 테이블 출력
short_names = [t[:15]+"..." for t in test_texts]
df_sim = pd.DataFrame(sim_matrix, columns=short_names, index=short_names)

print("유사도 행렬 (1.0 = 동일, 0.0 = 무관):")
display(df_sim.round(3).style.background_gradient(cmap='Blues', vmin=0, vmax=1))

In [ ]:
# 핵심 포인트: 의미가 비슷하면 유사도가 높다

pairs = [
    ("API 키를 안전하게 관리하는 방법", "시크릿 키 저장 및 보안 정책"),
    ("API 키를 안전하게 관리하는 방법", "CloudSync 파일 동기화 속도"),
    ("API 키를 안전하게 관리하는 방법", "점심 메뉴 추천해주세요"),
    ("재택근무 장비 지원 신청", "CloudSync 파일 동기화 속도"),
]

print("유사도 직관적 비교:")
print()
for a, b in pairs:
    vec_a = embed_texts([a])[0]
    vec_b = embed_texts([b])[0]
    sim = float(cosine_sim(vec_a, [vec_b])[0])
    bar_len = int(sim * 30)
    bar = "█" * bar_len + "░" * (30 - bar_len)
    print(f"  [{bar}] {sim:.3f}")
    print(f"  A: '{a[:30]}'")
    print(f"  B: '{b[:30]}'")
    print()

## 4️⃣ 벡터 인덱스 구축 (Vector Store)

실제 서비스에서는 Pinecone, Weaviate, pgvector 등을 씁니다.  
여기서는 개념 이해를 위해 numpy로 직접 구현합니다.

In [ ]:
class SimpleVectorStore:
    """강의용 초경량 벡터 스토어 (numpy 기반)."""

    def __init__(self):
        self.doc_ids = []
        self.contents = []
        self.embeddings = None
        self.metadata = []

    def build(self, documents):
        """문서를 인덱싱합니다."""
        print(f"📊 {len(documents)}개 문서 임베딩 중...")
        self.doc_ids = [d["doc_id"] for d in documents]
        self.contents = [d["content"] for d in documents]
        self.metadata = [{k:v for k,v in d.items() if k != "content"} for d in documents]
        self.embeddings = embed_texts(self.contents)
        print(f"✅ 인덱싱 완료! (차원: {self.embeddings.shape[1]})")

    def search(self, query, top_k=5):
        """유사 문서를 검색합니다."""
        query_vec = embed_texts([query])[0]
        scores = cosine_sim(query_vec, self.embeddings)
        top_idx = np.argsort(scores)[::-1][:top_k]
        return [
            {
                "doc_id": self.doc_ids[i],
                "content": self.contents[i],
                "score": float(scores[i]),
                "category": self.metadata[i].get("category", ""),
            }
            for i in top_idx
        ]

# 인덱스 구축
store = SimpleVectorStore()
store.build(SAMPLE_DOCS)

print(f"\n저장된 벡터 수: {len(store.doc_ids)}")
print(f"벡터 행렬 크기: {store.embeddings.shape}")

## 5️⃣ Vector Search 실행 - 쿼리별 결과 비교

In [ ]:
def show_search_results(query, results, title="벡터 검색 결과"):
    """검색 결과를 시각적으로 출력합니다."""
    rows = []
    for i, r in enumerate(results, 1):
        score_bar = "█" * int(r["score"] * 20) + "░" * (20 - int(r["score"] * 20))
        rows.append({
            "순위": f"#{i}",
            "점수": f"{r['score']:.4f}",
            "점수 막대": score_bar,
            "문서 ID": r["doc_id"],
            "카테고리": r["category"],
            "내용 (앞 70자)": r["content"][:70] + "..."
        })
    df = pd.DataFrame(rows)
    print(f"\n🔍 {title}")
    print(f"   쿼리: '{query}'")
    display(df)
    return df

# 다양한 쿼리로 테스트
queries = [
    "API 키를 안전하게 보관하는 방법",
    "새로 입사한 개발자가 첫 주에 해야 할 일",
    "CloudSync 최신 버전 변경사항",
    "서비스 장애 발생 시 대응 절차",
]

for query in queries:
    results = store.search(query, top_k=3)
    show_search_results(query, results)

## 6️⃣ BM25 vs Vector Search 비교

두 검색 방식의 차이를 직접 비교합니다.  
어떤 상황에서 어느 방식이 유리한지 보여줍니다.

In [ ]:
import re
from rank_bm25 import BM25Okapi

# BM25 인덱스 구축
def tokenize(text):
    """간단한 한국어 토크나이저."""
    tokens = re.sub(r'[^\w\s]', ' ', text).split()
    return [t.lower() for t in tokens if len(t) > 1]

contents = [d["content"] for d in SAMPLE_DOCS]
tokenized_docs = [tokenize(c) for c in contents]
bm25 = BM25Okapi(tokenized_docs)

def bm25_search(query, top_k=5):
    tokens = tokenize(query)
    scores = bm25.get_scores(tokens)
    top_idx = np.argsort(scores)[::-1][:top_k]
    return [
        {
            "doc_id": SAMPLE_DOCS[i]["doc_id"],
            "content": SAMPLE_DOCS[i]["content"],
            "score": float(scores[i]),
            "category": SAMPLE_DOCS[i]["category"],
        }
        for i in top_idx
    ]

print("✅ BM25 인덱스 구축 완료")

In [ ]:
# ─────────────────────────────────────────────────
# BM25가 강한 경우: 정확한 키워드 검색
# ─────────────────────────────────────────────────
q_keyword = "Python 3.10 최소 버전"  # 정확한 기술 키워드

bm25_res = bm25_search(q_keyword, top_k=3)
vec_res = store.search(q_keyword, top_k=3)

print("=" * 60)
print(f"  쿼리: '{q_keyword}'")
print("  (정확한 기술 버전 키워드 포함)")
print("=" * 60)

print("\n🔑 BM25 (키워드 검색) - 상위 3개:")
for i, r in enumerate(bm25_res, 1):
    print(f"  #{i} [{r['doc_id']}] 점수: {r['score']:.3f}")
    print(f"       {r['content'][:80]}...")

print("\n🧠 Vector Search (의미 검색) - 상위 3개:")
for i, r in enumerate(vec_res, 1):
    print(f"  #{i} [{r['doc_id']}] 점수: {r['score']:.4f}")
    print(f"       {r['content'][:80]}...")

print("\n💡 'Python 3.10'이라는 정확한 키워드는 BM25가 더 잘 찾습니다.")

In [ ]:
# ─────────────────────────────────────────────────
# Vector Search가 강한 경우: 의미 기반 검색
# ─────────────────────────────────────────────────
q_semantic = "키를 함부로 코드에 넣으면 안 되는 이유"  # 의미 이해 필요

bm25_res2 = bm25_search(q_semantic, top_k=3)
vec_res2 = store.search(q_semantic, top_k=3)

print("=" * 60)
print(f"  쿼리: '{q_semantic}'")
print("  (의미 이해가 필요한 쿼리)")
print("=" * 60)

print("\n🔑 BM25 (키워드 검색) - 상위 3개:")
for i, r in enumerate(bm25_res2, 1):
    print(f"  #{i} [{r['doc_id']}] 점수: {r['score']:.3f}")

print("\n🧠 Vector Search (의미 검색) - 상위 3개:")
for i, r in enumerate(vec_res2, 1):
    print(f"  #{i} [{r['doc_id']}] 점수: {r['score']:.4f}")
    print(f"       {r['content'][:80]}...")

print("\n💡 '코드에 넣으면 안 된다' = '하드코딩 금지'")
print("   단어는 다르지만 의미가 같다 → 벡터 검색이 더 잘 찾습니다.")

## 7️⃣ 검색 품질이 곧 답변 품질 - 시연

In [ ]:
# 중요한 케이스: 구버전 문서가 검색에 걸리는 문제
q_policy = "API 키 유효기간이 얼마나 되나요?"

results = store.search(q_policy, top_k=5)

print("=" * 60)
print(f"  쿼리: '{q_policy}'")
print("=" * 60)
print()

for i, r in enumerate(results, 1):
    emoji = "🚨" if "SEC-POL-002" in r["doc_id"] else "📄"
    print(f"  #{i} {emoji} [{r['doc_id']}] 점수: {r['score']:.4f}")
    # 유효기간 관련 부분만 하이라이트
    content = r['content']
    if "180일" in content:
        print(f"       ⚠️  내용에 '180일' 포함 (구버전 정책!)")
    elif "90일" in content or "365일" in content:
        print(f"       ✅ 내용에 '90일/365일' 포함 (현행 정책)")
    print(f"       {content[:80]}...")
    print()

print("🎯 핵심 포인트:")
print("   구버전 SEC-POL-002 (180일)와 현행 SEC-POL-003 (90일/365일)이 모두 검색됨.")
print("   검색 순위에 따라 LLM이 잘못된 정보를 답변할 수 있습니다.")
print("   → 이것이 retrieval quality가 중요한 이유입니다.")
print("   → 05번 노트북에서 이 문제를 더 다룹니다.")

In [ ]:
# 벡터 스토어 vs 실제 VectorDB 비교

display(HTML("""
<div style="font-family:Arial,sans-serif;max-width:780px;margin:10px auto;">
  <h3 style="color:#2c3e50;">실제 서비스에서는 어떻게 다른가?</h3>
  <table style="width:100%;border-collapse:collapse;font-size:13px;">
    <thead>
      <tr style="background:#2c3e50;color:white;">
        <th style="padding:10px;">항목</th>
        <th style="padding:10px;">이 데모 (numpy)</th>
        <th style="padding:10px;">실제 Vector DB</th>
      </tr>
    </thead>
    <tbody>
      <tr style="background:#f8f9fa;">
        <td style="padding:10px;">문서 수</td>
        <td style="padding:10px;">15개</td>
        <td style="padding:10px;">수백만~수억 개</td>
      </tr>
      <tr>
        <td style="padding:10px;">검색 방식</td>
        <td style="padding:10px;">전수 비교 (O(n))</td>
        <td style="padding:10px;">ANN 인덱스 (HNSW, IVF)</td>
      </tr>
      <tr style="background:#f8f9fa;">
        <td style="padding:10px;">영속성</td>
        <td style="padding:10px;">메모리 (재시작시 삭제)</td>
        <td style="padding:10px;">디스크 영속 저장</td>
      </tr>
      <tr>
        <td style="padding:10px;">메타데이터 필터</td>
        <td style="padding:10px;">수동 구현</td>
        <td style="padding:10px;">기본 지원</td>
      </tr>
      <tr style="background:#f8f9fa;">
        <td style="padding:10px;">예시 도구</td>
        <td style="padding:10px;">이 코드</td>
        <td style="padding:10px;">Pinecone, pgvector, Weaviate, Chroma</td>
      </tr>
    </tbody>
  </table>
  <div style="margin-top:12px;padding:10px;background:#e8f4fd;border-left:4px solid #3498db;font-size:13px;">
    💡 <strong>핵심 원리는 같습니다.</strong> 벡터 변환 → 유사도 계산 → 상위 k개 반환
  </div>
</div>
"""))

---

## 🎤 강의자 멘트 포인트

> **"Vector DB는 신비로운 AI 기술이 아닙니다.**  
> **'의미가 비슷한 문서를 빠르게 찾는 인프라'입니다.**  
>
> 여기서 중요한 건 LLM이 내 회사 문서를 학습하지 않았다는 점입니다.  
> Fine-tuning을 할 수도 있지만, 문서가 바뀔 때마다 재학습할 수는 없죠.  
> 그래서 검색 → LLM에 전달하는 방식이 실용적입니다.  
> 이게 RAG의 출발점입니다."

## 🙋 청중 질문 유도
> - "키워드 검색 vs 의미 검색, 여러분 서비스라면 어떤 게 더 중요할까요?"
> - "구버전 문서와 신버전 문서가 함께 있을 때, 어떻게 처리해야 할까요?"
> - "임베딩 벡터의 차원(384 vs 1536)이 검색 품질에 어떤 영향을 줄까요?"

## 🏗️ 실무 확장 포인트
- **메타데이터 필터링**: 날짜 범위, 카테고리별 검색 제한
- **청킹 전략**: 긴 문서를 어떻게 나눌지 (문단, 512토큰, 슬라이딩 윈도우)
- **다국어 임베딩**: `paraphrase-multilingual-MiniLM-L12-v2`
- **업데이트 전략**: 새 문서 추가 시 인덱스 갱신 방법

## ➕ 추가 실험 아이디어
1. 쿼리를 영어로 바꿔서 한국어 문서를 검색해보기 (다국어 임베딩)
2. 문서 내용을 완전히 다른 언어로 번역해서 검색 품질 비교
3. `top_k`를 1, 3, 10으로 바꿔서 어떤 문서가 추가로 들어오는지 확인

## ➡️ 다음 노트북
**04_basic_rag_demo.ipynb** - 검색한 문서를 LLM 답변에 연결하는 기본 RAG 구조